In [7]:
import pandas as pd
from pathlib import Path

clean_df = pd.read_csv("../data/clean_index.csv")
folds_df = pd.read_csv("../data/folds.csv")

for name, df in [("clean_index", clean_df), ("folds", folds_df)]:
    print("=" * 60)
    print(name, "| shape:", df.shape)
    print("Columns:", list(df.columns))
    print(df.head(3).to_string())

    # Show value counts for columns with few unique values (labels, folds, splits)
    for col in df.columns:
        if df[col].nunique() <= 10:
            print(f"\nValue counts for '{col}':")
            print(df[col].value_counts().to_string())

FileNotFoundError: [Errno 2] No such file or directory: '../data/clean_index.csv'

In [6]:
from pathlib import Path
import hashlib
import pandas as pd

DATA_DIR = Path("../data")
IMG_EXTS = {".jpg", ".jpeg", ".png"}


def folder_to_label(folder_name):
    """Map a raw folder name to a short label (order matters)."""
    name = folder_name.lower()
    if "history" in name:
        return "History_of_MI"
    if "abnormal" in name:
        return "Abnormal"
    if "normal person" in name:
        return "Normal"
    if "myocardial" in name:
        return "MI"
    return None  # ignore any other folder


rows = []
for folder in sorted(DATA_DIR.iterdir()):
    if not folder.is_dir():
        continue
    label = folder_to_label(folder.name)
    if label is None:
        continue
    for p in sorted(folder.iterdir()):
        if p.suffix.lower() in IMG_EXTS:
            rows.append(
                {
                    "path": str(p),
                    "label": label,
                    "hash": hashlib.md5(p.read_bytes()).hexdigest(),
                }
            )

raw_df = pd.DataFrame(rows)

# 1) Raw file count vs unique-content count per class
summary = raw_df.groupby("label").agg(
    total_files=("hash", "size"),
    unique_hashes=("hash", "nunique"),
)
print(summary)

# 2) Hashes that appear in more than one class (label conflicts)
classes_per_hash = raw_df.groupby("hash")["label"].nunique()
print("\nHashes shared across different classes:", (classes_per_hash > 1).sum())

# 3) Fold balance: images per class per fold
folds_df = pd.read_csv("../data/folds.csv")
print("\nImages per class per fold:")
print(pd.crosstab(folds_df["label"], folds_df["fold"]))

               total_files  unique_hashes
label                                    
Abnormal               233            233
History_of_MI          172             86
MI                     239             30
Normal                 284            142

Hashes shared across different classes: 0


FileNotFoundError: [Errno 2] No such file or directory: '../data/folds.csv'